# Where Does RL Learn? One Block, or One Percent, of Qwen3-1.7B

Reinforcement learning with verifiable rewards changes a model's behavior a great deal while, by several recent
accounts, changing surprisingly little of the model. Two papers make the claim precise in different ways:

- [Is One Layer Enough?](https://www.alphaxiv.org/abs/2607.01232) trains a single decoder block and finds that some
  blocks recover most of full training's gain on math benchmarks, and others almost none of it.
- [Understanding LLM Parameter Update Sparsity through the Lens of Fisher](https://www.alphaxiv.org/abs/2609.36262)
  trains only the 1% of coordinates with the largest Fisher information, and finds that it matches full training
  while 1% chosen at random does not.

Which parameters train is the variable in both, which makes them a natural fit for Fine-Tuning Scheduler: each
condition below is a one-phase schedule written in [venngram](https://github.com/speediedan/venngram) addresses
(`layers:10`, `role:base`), compiled against the model's view, and the two mask conditions add a gradient mask to the
same schedule.

**What is reproduced, transferred and new.** Everything runs on Qwen3-1.7B-Base, the Fisher paper's own model, with
the Fisher paper's recipe (its Appendix B.2). The full and Fisher 1% arms reproduce that paper's comparison, and the
random 1% arm is its control. The single-block arms carry the One Layer contrast over to the Fisher paper's recipe, a
combination neither paper ran: One Layer trains at ten times the learning rate, on different data, with longer
responses.

**What this notebook runs.** The five training runs took about 47 hours on one RTX 4090 and are not repeated here.
This notebook rebuilds each arm's schedule on a weightless copy of the model (only its configuration is downloaded),
checks what each one selects, and analyzes the runs' measured results, which ship with the examples
(`fts_examples/where_rl_learns/results/seed0.json`). The last section gives the commands that produced them.

## Setup

In [ ]:
# the model whose configuration the schedules are compiled against, and the results to analyze (None: the packaged
# results of the five arms, training seed 0)
MODEL_ID = "Qwen/Qwen3-1.7B-Base"
MODEL_REVISION = "ea980cb0a6c2ae4b936e82123acc929f1cec04c1"
RESULTS = None

In [ ]:
import warnings

import torch
from transformers import AutoConfig, AutoModelForCausalLM

from finetuning_scheduler.core.compile import compile_schedule
import venngram

from fts_examples.tunable_views import use_seed_registry
from fts_examples.where_rl_learns.compare import compare, load_summary, RESULTS_DIR
from fts_examples.where_rl_learns.pilot import LEARNING_RATE, PROMPTS_PER_UPDATE, RESPONSES_PER_PROMPT
from fts_examples.where_rl_learns.train import ARMS, EVAL_SEEDS, expected_trainable, MASK_ARMS

warnings.filterwarnings("ignore", message=".*addresses.*experimental")  # said once per process
print("component maps from:", *use_seed_registry())

## The Recipe

Every arm trains with the same recipe, the Fisher paper's: Dr. GRPO (each response's advantage is its reward minus
its group's mean, without GRPO's per-response length and standard-deviation normalization) on SimpleRL-Zoo's Abel-format MATH problems at levels 3 to 5, with a
binary reward from an automatic answer check (`math-verify`). Nothing is judged by a model.

| setting | value |
| --- | --- |
| model | Qwen3-1.7B-Base, full precision for the policy |
| updates | 1,000, one PPO epoch per batch |
| batch | 4 prompts, 8 responses each, a 4,096-token horizon |
| optimizer | AdamW, learning rate $5 \times 10^{-7}$, gradients clipped at norm 1 |
| evaluation | MATH-500, 4 responses per problem (seeds 300 to 303), mean response accuracy, every 250 updates |

Sampling runs in vLLM, in its own process, with the policy's weights reloaded after every update and checked against
the policy's own log probabilities before training and after every 125 updates.

In [ ]:
print(f"learning rate {LEARNING_RATE}, {PROMPTS_PER_UPDATE} prompts x {RESPONSES_PER_PROMPT} responses per update, "
      f"evaluation seeds {list(EVAL_SEEDS)}")

## Five Arms, Five Schedules

Each arm is a one-phase schedule: the phase's `params` are venngram addresses, resolved through the model's view when
the schedule compiles. `role:base` is every tensor of the model, and `layers:10` is the eleventh decoder block,
whatever the architecture calls its tensors. The mask arms schedule the whole model and then train only a mask's
coordinates.

`AutoModelForCausalLM.from_config` under `torch.device("meta")` builds the module tree without allocating any weights,
which is all a schedule needs to compile.

In [ ]:
with torch.device("meta"):
    model = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION))
view = venngram.view(model)
params = dict(model.named_parameters())
total = sum(p.numel() for p in params.values())
print(f"{type(model).__name__}: {len(params)} tensors, {total / 1e9:.2f}B parameters, map {view.cmap.architecture}")

for arm, addresses in ARMS.items():
    selected = compile_schedule({0: {"params": addresses}}, view).phases[0].physical
    assert sorted(selected) == sorted(expected_trainable(arm, list(params))), arm
    n = sum(params[name].numel() for name in selected)
    mask = f", then a {MASK_ARMS[arm]} mask" if arm in MASK_ARMS else ""
    print(f"{arm:<8} {' | '.join(addresses):<10} {len(selected):>4} tensors, {n / 1e6:8.1f}M parameters "
          f"({100 * n / total:5.1f}%){mask}")

Each single-block arm trains 11 tensors and 50.3M parameters, 2.9% of the model; the assertion checks that each
address selected exactly the block its name says.

**The masks.** A mask arm keeps a fixed 1% of the model's coordinates, about 17.2M, and zeroes the gradient of every
other coordinate after the gradients accumulate and before clipping, as the Fisher paper does. The Fisher mask keeps
the coordinates with the largest diagonal Fisher information, estimated over 800 training prompts on the untrained
model; the random mask keeps the same number chosen uniformly. Both are global selections over the whole model (the
masks are 69 MB each and are built rather than shipped, see the last section). Every run checks, at every
checkpoint, that coordinates outside its mask still hold their pretrained values exactly.

In [ ]:
arms = load_summary(RESULTS) if RESULTS else load_summary()
for name in MASK_ARMS:
    mask = arms[name]["run"]["mask"]
    print(f"{name}: {mask['kind']} mask, {mask['kept']:,} of {mask['coordinates']:,} coordinates "
          f"({100 * mask['kept'] / mask['coordinates']:.2f}%), digest {mask['digest'][:12]}")

## What Each Arm Learned

The evaluation scores every response, so each arm's accuracy is known problem by problem: the summary keeps each of
the 500 problems' count of correct responses (out of 4) at every evaluation.

In [ ]:
ORDER = ["full", "fisher1", "layer10", "random1", "layer24"]
updates = [e["update"] for e in arms["full"]["evals"]]
print(f"{'arm':<8}" + "".join(f"{u:>8}" for u in updates))
for name in ORDER:
    print(f"{name:<8}" + "".join(f"{e['accuracy_percent']:8.1f}" for e in arms[name]["evals"]))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for name in ORDER:
    evals = arms[name]["evals"]
    ax.errorbar([e["update"] for e in evals], [e["accuracy_percent"] for e in evals],
                yerr=[e["standard_error_percent"] for e in evals], marker="o", capsize=3, label=name)
ax.set_xlabel("update")
ax.set_ylabel("MATH-500 accuracy (%)")
ax.set_title("Qwen3-1.7B-Base, Dr. GRPO, one training seed")
ax.legend()
plt.show()

## Paired Differences

Every arm answers the same 500 problems with the same sampling seeds, so two arms compare problem by problem. The
difference of their per-problem accuracies has a much smaller standard error than either accuracy alone, because most
of the variation between problems is shared. `compare` gives each arm's gain over its own start and every pair's
difference at the last evaluation.

In [ ]:
result = compare({name: arms[name] for name in ORDER})
full_gain = result["gain_over_initial"]["full"]["difference_points"]
print(f"gain over its own start at update {result['update']}, in points (share of full training's gain):")
for name in ORDER:
    g = result["gain_over_initial"][name]
    print(f"  {name:<8} {g['difference_points']:+6.2f} +- {g['standard_error_points']:.2f} "
          f"({100 * g['difference_points'] / full_gain:5.1f}%)")
print("pairs, in points:")
for a, b in zip(ORDER, ORDER[1:]):
    p = result["pairs"][f"{a} - {b}"]
    print(f"  {a + ' - ' + b:<18} {p['difference_points']:+6.2f} +- {p['standard_error_points']:.2f}")
p = result["pairs"]["fisher1 - random1"]
print(f"  {'fisher1 - random1':<18} {p['difference_points']:+6.2f} +- {p['standard_error_points']:.2f}")

The training reward tells the same story from the other side: it is the fraction of the training batch's responses
that the answer check accepts, so an arm whose reward does not move did not learn the training task at all, not only
failed to carry it over to MATH-500.

In [ ]:
for name in ORDER:
    rewards = arms[name]["reward_per_50_updates"]
    first, last = sum(rewards[:4]) / 4, sum(rewards[-4:]) / 4
    print(f"{name:<8} mean reward over the first 200 updates {first:.3f}, over the last 200 {last:.3f}")

## What This Shows

On Qwen3-1.7B-Base, under the Fisher paper's recipe:

- **Reproduced.** 1% of the coordinates, chosen by Fisher information, recovers about 97% of full training's gain
  (2 points behind it, about two standard errors), while the same budget chosen at random recovers about a fifth of
  it, 28 points behind the Fisher mask. Which coordinates train matters far more than how few.
- **New.** A single middle block (layer 10, 2.9% of the parameters) recovers about 73% of the gain; a single late
  block (layer 24) recovers almost none, and its training reward never moves. The block matters as much as the
  budget. One Layer reports late blocks as weaker on this model; at the Fisher recipe's learning rate the late block
  does not move at all within 1,000 updates, and whether it would at One Layer's rate is untested here.
- **The ordering** `full`, `fisher1`, `layer10`, `random1`, `layer24` holds with every adjacent gap several standard
  errors wide, except the first.

What these runs cannot say:

- **One training seed per arm.** The standard errors are over problems and evaluation seeds, not over training runs,
  and identical runs do not repeat exactly (vLLM sampling is not batch invariant), so the spread between runs of one
  arm is unmeasured. The gaps above are large against any plausible spread; the two-point gap between `full` and
  `fisher1` is not.
- **1,000 updates.** `layer10` and `random1` are still rising at the end, so their gaps may close with longer
  training.

## Running an Arm

Each arm ran overnight on one 24 GB GPU (about 7 to 10 hours). The trainer and vLLM run in separate environments, because
vLLM pins its own torch; the trainer holds the full-precision policy and its optimizer state in host memory while vLLM
samples, so allow about 40 GB of host memory.

```bash
# build the masks once (the Fisher estimate takes about 80 minutes; the random mask a few seconds)
python -m fts_examples.where_rl_learns.masks --kind fisher --output masks/fisher1.pt --sampler vllm \
    --vllm-python /path/to/vllm/bin/python
python -m fts_examples.where_rl_learns.masks --kind random --output masks/random1_seed0.pt

# one arm; rerunning the same command resumes from its last checkpoint
python -m fts_examples.where_rl_learns.train --arm layer10 --output-dir runs/layer10 --sampler vllm \
    --vllm-python /path/to/vllm/bin/python
python -m fts_examples.where_rl_learns.train --arm fisher1 --mask masks/fisher1.pt --output-dir runs/fisher1 \
    --sampler vllm --vllm-python /path/to/vllm/bin/python

# compare runs, or write them as a summary this notebook can read (RESULTS above)
python -m fts_examples.where_rl_learns.compare runs/full runs/layer10 runs/fisher1
python -m fts_examples.where_rl_learns.compare runs/* --summarize my_results.json
```